# Exploración de la reconstrucción de los autoencoders

Plantilla del change `ae-reconstruccion` (tarea 8.4). **Nivel 1: sin TensorFlow.**
Lee `reconstruction.parquet`, `diagnostics.parquet` y las métricas del run de AE
desde el estado de MLflow (`analysis/load.py`, sin duplicar la carga).

Diagnósticos de aplanamiento: razón de varianza, pendiente real vs reconstruido,
error por nivel de concentración, y comparación contra `ref-mean` y `ref-pca-d*`.

Gráficos con matplotlib (no Plotly: en el Space las figuras interactivas tardan demasiado).

In [ ]:
import sys, pathlib
# Raíz de importación del pipeline: el notebook vive en notebooks/ (o en la raíz del repo).
_here = pathlib.Path.cwd()
PIPELINE = next(p for p in (_here / "pipeline", _here.parent / "pipeline") if p.exists())
sys.path.insert(0, str(PIPELINE))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from analysis import load
from features.build import load_validated
from ae import data as aed
from ae import dmd as dmdm
from ae import metrics as aem

BUCKET_DATA = "sinca-data"
# Credenciales: el rol del Space (sinca-dev-role) las provee; no escribir claves aquí.

## 1. Estado de MLflow y runs de autoencoder

In [ ]:
state = load.ensure_state()  # S3 real (sinca-mlflow/_mlflow); o load.ensure_state(data_dir=<fixture>)
runs = load.list_runs(state)
ae_runs = load.ae_runs(runs)
refs = [r for r in runs if load.is_reference_run(r)]
print("AE:", [r["run_name"] for r in ae_runs])
print("Referencias:", [r["run_name"] for r in refs])

## 2. Elegir un run y cargar sus artefactos

In [ ]:
RUN_NAME = ae_runs[-1]["run_name"]  # cambiar a mano para comparar otro run
run = next(r for r in ae_runs if r["run_name"] == RUN_NAME)
arts = load.load_artifacts(state, run["run_id"])
recon, diag = arts["reconstruction.parquet"], arts["diagnostics.parquet"]
print("compuerta:", run["metrics"].get("gate_pass"), "| supera PCA:", run["metrics"].get("gate_beats_pca"))

## 3. Real vs reconstruido con zoom a un episodio

In [ ]:
serie = "Nielol|pm25"
frame = recon[recon["serie"] == serie].sort_values("timestamp")
ts = frame["timestamp"]
frame = frame[(ts >= pd.Timestamp("2026-07-01", tz=ts.dt.tz)) & (ts <= pd.Timestamp("2026-07-15", tz=ts.dt.tz))]
fig, ax = plt.subplots(figsize=(12, 3.8))
ax.plot(frame["timestamp"], frame["y_true"], lw=1.1, color="#1f2937", label="real")
ax.plot(frame["timestamp"], frame["y_recon"], lw=1.1, color="#dc2626", label="reconstruido")
ax.set_title(f"Reconstrucción de MP — {serie} (µg/m³)")
ax.set_ylabel("µg/m³")
ax.legend()
fig.tight_layout()
plt.show()

## 4. Aplanamiento y error por nivel de concentración

In [ ]:
display(diag.pivot_table(index="serie", columns="metric", values="value").round(3))
test = diag[diag["split"] == "test"]

# Aplanamiento: razón de varianza y pendiente por serie (1 = ideal)
flat = test[test["metric"].isin(["var_ratio", "slope"])].pivot_table(index="serie", columns="metric", values="value", aggfunc="first")
ax = flat.plot.bar(figsize=(10, 3.5), color={"var_ratio": "#dc2626", "slope": "#1f2937"}, title="Aplanamiento (test)")
ax.axhline(1.0, ls=":", color="#9ca3af")
plt.tight_layout()
plt.show()

# Error por nivel de concentración (RMSE por tramo, µg/m³)
rows = []
for serie, group in test.groupby("serie"):
    vals = dict(zip(group["metric"], group["value"]))
    for b in ("lt50", "50_100", "gt100"):
        if f"rmse_{b}" in vals:
            rows.append({"serie": serie, "tramo": b, "rmse": vals[f"rmse_{b}"]})
err = pd.DataFrame(rows).pivot(index="serie", columns="tramo", values="rmse")
err.plot.bar(figsize=(10, 3.5), title="Error por nivel de concentración (test)", ylabel="µg/m³")
plt.tight_layout()
plt.show()

## 5. Contra las referencias y ablación de z_aux

In [ ]:
rows = [{"run": run["run_name"], **load.ae_metric(run, "recon_rmse_mean_")}]
rows += [{"run": r["run_name"], "test": r["metrics"].get("recon_rmse_mean_test"), "valid": r["metrics"].get("recon_rmse_mean_valid")} for r in refs]
display(pd.DataFrame(rows))
print("Δ RMSE al anular z_aux (test):", run["metrics"].get("ablation_zaux_delta_rmse_test"))